### 1. Bibliotecas

In [1]:
import pandas as pd
import numpy as np
import random
import os
import json
import matplotlib.pyplot as plt
import seaborn as sns
import re
from datetime import datetime, timedelta

In [2]:
print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Matplotlib: {plt.matplotlib.__version__}")
print(f"Seaborn: {sns.__version__}")

Pandas: 3.0.3
NumPy: 2.4.6
Matplotlib: 3.11.0
Seaborn: 0.13.2


### 2. Dataset e informações básicas

In [2]:
# criar dataset sintético com 200 registros de vendas e com alguns dados sujos

def gerar_dataset_vendas(n_registros=200, seed=1):
    random.seed(seed)
    np.random.seed(seed)
    produtos = ['Notebook', 'Smartphone', 'Tablet', 'Monitor', 'Teclado', 'Mouse']
    precos = { 'Notebook': 3500, 'Smartphone': 2200, 'Tablet': 1800,
               'Monitor': 1200, 'Teclado': 250, 'Mouse': 120 }
    categorias = { "Notebook": "Computadores", "Smartphone": "Celulares",
                   "Tablet": "Celulares", "Monitor": "Computadores", "Teclado": "Periféricos",
                   "Mouse": "Periféricos" }
    regioes = ["Sudeste", "Sul", "Nordeste", "Centro-Oeste", "Norte"]
    clientes = [f"Cliente_{i:03d}" for i in range(1, 31)]
    
    data_inicio = datetime(2024, 1, 1)
    dados = []

    for i in range(n_registros):
        produto = random.choice(produtos)
        quantidade = random.randint(1, 10)
        preco = precos[produto]
        data = data_inicio + timedelta(days=random.randint(0, 364))

        # Inserindo dados sujos para limpeza
        if random.random() < 0.05:
            quantidade = None 
        if random.random() < 0.04:
            preco = None 
        if random.random() < 0.03:
            produto = " " + produto 
        
        data_str = data.strftime("%Y-%m-%d") if random.random() > 0.02 else "DATA INVALIDA"
        dados.append({
            "id_venda": i + 1,
            "data_venda": data_str,
            "cliente": random.choice(clientes),
            "produto": produto,
            "categoria": categorias.get(produto.strip(), "Outros"),
            "regiao": random.choice(regioes),
            "quantidade": quantidade,
            "preco_unitario": preco
        })
    
    return pd.DataFrame(dados)

In [3]:
# exibir informações básicas do DataFrame

def inspecionar_dados(df):
    print("\n=== INSPEÇÃO INICIAL DO DATASET ===")
    print(f"Shape: {df.shape}")
    print(f"\nColunas: {list(df.columns)}")
    print(f"\nTipos de dados:\n{df.dtypes}")
    print(f"\nValores nulos por coluna:\n{df.isnull().sum()}")
    print(f"\nPrimeiros registros:\n{df.head()}")
    print(f"\nEstatísticas descritivas:\n{df.describe()}")

In [4]:
# Gerando o dataset (df_bruto) e salvando em /data/raw

df_bruto = gerar_dataset_vendas()
df_bruto.to_csv("../data/raw/vendas.csv", index=False)
print(f"Dataset gerado com {len(df_bruto)} registros.")
df_bruto.head() 

Dataset gerado com 200 registros.


,id_venda,data_venda,cliente,produto,categoria,regiao,quantidade,preco_unitario
0,1,2024-02-02,Cliente_026,Smartphone,Celulares,Sul,10.0,2200.0
1,2,2024-01-15,Cliente_023,Notebook,Computadores,Centro-Oeste,8.0,3500.0
2,3,2024-10-29,Cliente_018,Tablet,Celulares,Sudeste,4.0,1800.0
3,4,2024-08-04,Cliente_018,Monitor,Computadores,Sul,4.0,1200.0
4,5,2024-12-12,Cliente_027,Tablet,Celulares,Norte,4.0,1800.0


In [5]:
# informações básicas do DataFrame

inspecionar_dados(df_bruto)


=== INSPEÇÃO INICIAL DO DATASET ===
Shape: (200, 8)

Colunas: ['id_venda', 'data_venda', 'cliente', 'produto', 'categoria', 'regiao', 'quantidade', 'preco_unitario']

Tipos de dados:
id_venda            int64
data_venda            str
cliente               str
produto               str
categoria             str
regiao                str
quantidade        float64
preco_unitario    float64
dtype: object

Valores nulos por coluna:
id_venda           0
data_venda         0
cliente            0
produto            0
categoria          0
regiao             0
quantidade        11
preco_unitario     5
dtype: int64

Primeiros registros:
   id_venda  data_venda      cliente     produto     categoria        regiao  \
0         1  2024-02-02  Cliente_026  Smartphone     Celulares           Sul   
1         2  2024-01-15  Cliente_023    Notebook  Computadores  Centro-Oeste   
2         3  2024-10-29  Cliente_018      Tablet     Celulares       Sudeste   
3         4  2024-08-04  Cliente_018     Mon

### 3. Limpeza dos dados

In [6]:
# limpar espaços em colunas 

def limpar_colunas(df, colunas):
    df = df.copy() 
    for col in colunas:
        df[col] = df[col].apply(
        lambda s: re.sub(r"\s+", " ", str(s)).strip() if pd.notna(s) else s)
        return df

In [7]:
# efetuar limpeza dos dados em 4 etapas (colunas, datas, nulos, números (quantidade e preço))

def limpar_dados(df):
    df = df.copy()
    n_inicial = len(df)
    relatorio = {}
        
    colunas_texto = df.select_dtypes(include=["object","string"]).columns
    df = limpar_colunas(df, colunas_texto)
    
    df["data_venda"] = pd.to_datetime(df["data_venda"], errors="coerce")
    relatorio["datas_invalidas_removidas"] = int(df["data_venda"].isnull().sum())
    df = df.dropna(subset=["data_venda"])

    n_antes = len(df)
    df = df.dropna(subset=["quantidade", "preco_unitario"])
    relatorio["linhas_nulas_removidas"] = n_antes - len(df)

    df["quantidade"] = df["quantidade"].astype(int)
    df["preco_unitario"] = df["preco_unitario"].astype(float)

    relatorio["registros_iniciais"] = n_inicial
    relatorio["registros_finais"] = len(df)
    relatorio["registros_removidos_total"] = n_inicial - len(df) 
    print("=== RELATORIO DE LIMPEZA ===")
    for k, v in relatorio.items():
        print(f" {k}: {v}")
    
    return df, relatorio

In [8]:
# limpando o dataset e criando a versão df_v1 que será salva como vendas_v1.csv (com outliers)

df_v1, relatorio = limpar_dados(df_bruto)
df_v1.to_csv("../data/processed/v1_com_outliers/vendas_v1.csv", index=False)
print("\nArquivo vendas_v1.csv salvo em v1_com_outliers")

=== RELATORIO DE LIMPEZA ===
 datas_invalidas_removidas: 4
 linhas_nulas_removidas: 16
 registros_iniciais: 200
 registros_finais: 180
 registros_removidos_total: 20

Arquivo vendas_v1.csv salvo em v1_com_outliers


### 4. Detecção e tratamento de outliers

In [9]:
# tratar outliers de colunas numéricas usando o Intervalo Interquartil (IQR)
# metodo = sem especificar ou remover -> remove abaixo e acima dos limites inferior e superior respectivamente
# metodo = outro valor ->  winsorização, altera os valores extremos pelos limites inferior ou superior

def tratar_outliers(df, colunas, fator=1.5, metodo='remover'):
    df = df.copy() 
    for col in colunas:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1 
        lim_inf = q1 - fator * iqr 
        lim_sup = q3 + fator * iqr 

        n_out = ((df[col] < lim_inf) | (df[col] > lim_sup)).sum()
        print(f' {col}: {n_out} outliers detectados '
              f'(lim_inf={lim_inf:.2f}, lim_sup={lim_sup:.2f})')
        
        if metodo == 'remover':
            df = df[(df[col] >= lim_inf) & (df[col] <= lim_sup)]
        else:
            df[col] = df[col].clip(lower=lim_inf, upper=lim_sup)
    return df

In [10]:
# criando uma versão temporária (df_v1_tmp) de df_v1 com a coluna receita_total (quantidade * preco_unitario)
# tratando os outliers em df_v1_tmp que será atribuído a df_v2 e salvo como vendas_v2.csv sem a coluna receita_total

df_v1_tmp = df_v1.copy()
df_v1_tmp["receita_total"] = df_v1_tmp["quantidade"] * df_v1_tmp["preco_unitario"]

df_v2 = tratar_outliers(df_v1_tmp, colunas=["quantidade", "receita_total"], metodo='remover')

df_v2 = df_v2.drop(columns=["receita_total"])

print(f"\nv1 = {len(df_v1)} linhas (com outliers)")
print(f"v2 = {len(df_v2)} linhas (outliers removidos)")
print(f"Diferença = {len(df_v1) - len(df_v2)} linhas removidas")

os.makedirs("data/v2_outliers_tratado", exist_ok=True)
df_v2.to_csv("data/v2_outliers_tratado/vendas_v2.csv", index=False)
print("\nArquivo vendas_v2.csv salvo em v2_outliers_tratado/")

 quantidade: 0 outliers detectados (lim_inf=-2.62, lim_sup=14.38)
 receita_total: 9 outliers detectados (lim_inf=-15000.00, lim_sup=28200.00)

v1 = 180 linhas (com outliers)
v2 = 171 linhas (outliers removidos)
Diferença = 9 linhas removidas

Arquivo vendas_v2.csv salvo em v2_outliers_tratado/


### 5. Criação de colunas derivadas

In [11]:
# criar colunas derivadas
#   novamente a coluna receita_total (quantidade * preco_unitario)
#   mes / trimestre / ano
#   faixa_receita_item (classificação do valor de venda)

def criar_colunas_derivadas(df):
    df = df.copy()
    df["receita_total"] = df["quantidade"] * df["preco_unitario"]
    df["mes"] = df["data_venda"].dt.month
    df["trimestre"] = df["data_venda"].dt.quarter.apply(lambda q: f"Q{q}")
    df["ano"] = df["data_venda"].dt.year

    condicoes = [
        (df["receita_total"] < 500), (df["receita_total"] >= 500) & 
        (df["receita_total"] < 5000), (df["receita_total"] >= 5000)
    ]
    rotulos = ["Baixo Valor", "Médio Valor", "Alto Valor"] 
    df["faixa_receita_item"] = np.select(condicoes, rotulos, default="N/D")

    print("COLUNAS DERIVADAS CRIADAS")
    return df

In [12]:
# criando as colunas derivadas e atribuindo ao df que será o DataFrame a ser analisado

df = criar_colunas_derivadas(df_v2)
df[["data_venda", "receita_total", "mes", "trimestre", "faixa_receita_item"]].head()

COLUNAS DERIVADAS CRIADAS


,data_venda,receita_total,mes,trimestre,faixa_receita_item
0,2024-02-02,22000.0,2,Q1,Alto Valor
1,2024-01-15,28000.0,1,Q1,Alto Valor
2,2024-10-29,7200.0,10,Q4,Alto Valor
3,2024-08-04,4800.0,8,Q3,Médio Valor
4,2024-12-12,7200.0,12,Q4,Alto Valor


### 6. Cálculo e exibição de métricas agrupadas

In [13]:
# calcular e exibir métricas agrupadas por: mês, produto, categoria e região
#   receita total e quantidade vendida por mês;
#   receita total por produto (top 5);
#   receita total por categoria;
#   receita total por região.

def calcular_metricas(df):
    metricas = {}
    
    metricas["por_mes"] = (df.groupby("mes").agg(
                           receita_total=("receita_total", "sum"), # soma da receita no mês
                           quantidade=("quantidade", "sum"), # total de itens vendidos
                           n_vendas=("id_venda", "count"), # número de transações  
                           ).reset_index().sort_values("mes")
                           )
    
    metricas["top_produtos"] = (df.groupby("produto")["receita_total"]
                                .sum()
                                .sort_values(ascending=False)
                                .head(5)
                                .reset_index()
                                )
    
    metricas["por_categoria"] = (df.groupby("categoria")["receita_total"]
                                 .sum()
                                 .reset_index()
                                 .sort_values("receita_total", ascending=False)
                                )
    
    metricas["por_regiao"] = (df.groupby("regiao")
                              .agg(
                              receita_total=("receita_total", "sum"),
                              media_ticket=("receita_total", "mean"),
                              n_vendas=("id_venda", "count")
                              )
                              .reset_index()
                              .sort_values("receita_total", ascending=False)
                             )
    
    for nome, tabela in metricas.items():
        print(f"\n=== {nome.upper().replace('_', ' ')} ===")
        print(tabela.to_string(index=False))
    return metricas

In [14]:
# calculando e exibindo as métricas agrupadas

metricas = calcular_metricas(df)


=== POR MES ===
 mes  receita_total  quantidade  n_vendas
   1       150650.0         104        16
   2       118290.0          84        17
   3        60310.0          58        11
   4        74560.0          72        12
   5        36490.0          47         9
   6       102060.0          65        12
   7       125790.0         123        24
   8       109310.0          87        18
   9        52650.0          87        17
  10       100590.0          79        11
  11       178900.0          85        15
  12        59870.0          59         9

=== TOP PRODUTOS ===
   produto  receita_total
Smartphone       347600.0
  Notebook       304500.0
   Monitor       229200.0
    Tablet       210600.0
   Teclado        47500.0

=== POR CATEGORIA ===
   categoria  receita_total
   Celulares       558200.0
Computadores       538500.0
 Periféricos        72770.0

=== POR REGIAO ===
      regiao  receita_total  media_ticket  n_vendas
       Norte       271010.0   6159.318182        44
